# Knowledge Distillation
------------------------

Welcome, during this ungraded lab you are going to perform a model compression technique known as **knowledge distillation** in which a `student` model "learns" from a more complex model known as the `teacher`. In particular you will:

1. Define a `Distiller` class with the custom logic for the distillation process.
2. Train the `teacher` model — a CNN with **BatchNormalization** and dropout regularization trained on **CIFAR-10**.
3. Train a `student` model (a smaller version of the teacher without regularization) using knowledge distillation.
4. Train another `student` model from scratch without distillation called `student_scratch`.
5. Compare the three models.

**Modifications from the original lab:**
- Dataset changed from *Cats vs Dogs* (binary, 224×224, ~787 MB download) to **CIFAR-10** (10 classes, 32×32, built into Keras — no download needed).
- Teacher model updated to include **BatchNormalization** layers for more stable training.
- Student model updated to two convolutional layers to handle the 10-class problem.
- Distillation temperature raised to **10** (produces softer probability distributions, more useful with 10 classes).
- Alpha set to **0.1** (90% distillation loss, 10% student cross-entropy loss).

This notebook is based on [this](https://keras.io/examples/vision/knowledge_distillation/) official Keras tutorial.

If you want a more theoretical approach to this topic be sure to check this paper [Hinton et al. (2015)](https://arxiv.org/abs/1503.02531).

Let's get started!

## Imports

In [ ]:
# For setting random seeds
import os
os.environ['PYTHONHASHSEED'] = str(42)

# Libraries
import random
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt

# More random seed setup
tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

print(f"TensorFlow version: {tf.__version__}")

## Prepare the data

For this lab you will use **CIFAR-10**, a classic benchmark dataset containing **60,000 colour images** (32×32 pixels) across **10 classes**:

> airplane · automobile · bird · cat · deer · dog · frog · horse · ship · truck

CIFAR-10 is built directly into Keras — no external download required. Begin by loading and splitting it:

In [ ]:
# Load CIFAR-10 — already split into 50,000 train and 10,000 test images
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.cifar10.load_data()

# Squeeze label arrays from shape (N, 1) to (N,) for SparseCategoricalCrossentropy
y_train_full = y_train_full.squeeze()
y_test = y_test.squeeze()

NUM_CLASSES = 10
CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print(f"Full training set : {x_train_full.shape}")
print(f"Test set          : {x_test.shape}")
print(f"Number of classes : {NUM_CLASSES}")

Preprocess the data by normalizing pixel values to `[0, 1]` and creating a validation split (10% of training data):

In [ ]:
# Normalize pixel values to [0, 1]
x_train_full = x_train_full.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Carve out 10% of training data as validation set
VAL_SIZE = 5000
x_val, y_val = x_train_full[:VAL_SIZE], y_train_full[:VAL_SIZE]
x_train, y_train = x_train_full[VAL_SIZE:], y_train_full[VAL_SIZE:]

BATCH_SIZE = 64

# Build tf.data pipelines
train_batches = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(45000)
    .batch(BATCH_SIZE)
    .prefetch(1)
)
validation_batches = (
    tf.data.Dataset.from_tensor_slices((x_val, y_val))
    .batch(BATCH_SIZE)
    .prefetch(1)
)
test_batches = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .batch(1)
)

print(f"Training samples   : {len(x_train)}")
print(f"Validation samples : {len(x_val)}")
print(f"Test samples       : {len(x_test)}")

## Code the custom `Distiller` model

In order to implement the distillation process you will create a custom Keras model which you will name `Distiller`. In order to do this you need to override some of the vanilla methods of a `keras.Model` to include the custom logic for the knowledge distillation. You need to override these methods:
- `compile`: This model needs some extra parameters to be compiled such as the teacher and student losses, the alpha and the temperature.
- `train_step`: Controls how the model is trained. This will be where the actual knowledge distillation logic will be found. This method is what is called when you do `model.fit`.
- `test_step`: Controls the evaluation of the model. This method is what is called when you do `model.evaluate`.

To learn more about customizing models check out the [official docs](https://keras.io/guides/customizing_what_happens_in_fit/).

In [ ]:
class Distiller(keras.Model):

  # Needs both the student and teacher models to create an instance of this class
  def __init__(self, student, teacher):
      super(Distiller, self).__init__()
      self.teacher = teacher
      self.student = student


  # Will be used when calling model.compile()
  def compile(self, optimizer, metrics, student_loss_fn,
              distillation_loss_fn, alpha, temperature):

      # Compile using the optimizer and metrics
      super(Distiller, self).compile(optimizer=optimizer, metrics=metrics)

      # Add the other params to the instance
      self.student_loss_fn = student_loss_fn
      self.distillation_loss_fn = distillation_loss_fn
      self.alpha = alpha
      self.temperature = temperature


  # Will be used when calling model.fit()
  def train_step(self, data):
      # Data is expected to be a tuple of (features, labels)
      x, y = data

      # Vanilla forward pass of the teacher
      # Note that the teacher is NOT trained
      teacher_predictions = self.teacher(x, training=False)

      # Use GradientTape to save gradients
      with tf.GradientTape() as tape:
          # Vanilla forward pass of the student
          student_predictions = self.student(x, training=True)

          # Compute vanilla student loss
          student_loss = self.student_loss_fn(y, student_predictions)

          # Compute distillation loss
          # KL divergence between logits softened by a temperature factor
          distillation_loss = self.distillation_loss_fn(
              tf.nn.softmax(teacher_predictions / self.temperature, axis=1),
              tf.nn.softmax(student_predictions / self.temperature, axis=1))

          # Compute loss by weighting the two previous losses using the alpha param
          loss = self.alpha * student_loss + (1 - self.alpha) * distillation_loss

      # Use tape to calculate gradients for student
      trainable_vars = self.student.trainable_variables
      gradients = tape.gradient(loss, trainable_vars)

      # Update student weights
      # Note that this is done ONLY for the student
      self.optimizer.apply_gradients(zip(gradients, trainable_vars))

      # Update the metrics
      self.compiled_metrics.update_state(y, student_predictions)

      # Return a performance dictionary
      results = {m.name: m.result() for m in self.metrics}
      results.update({"student_loss": student_loss, "distillation_loss": distillation_loss})
      return results


  # Will be used when calling model.evaluate()
  def test_step(self, data):
      # Data is expected to be a tuple of (features, labels)
      x, y = data

      # Use student to make predictions
      y_prediction = self.student(x, training=False)

      # Calculate student's vanilla loss
      student_loss = self.student_loss_fn(y, y_prediction)

      # Update the metrics
      self.compiled_metrics.update_state(y, y_prediction)

      # Return a performance dictionary
      results = {m.name: m.result() for m in self.metrics}
      results.update({"student_loss": student_loss})
      return results


## Teacher and student models

For the models you will use CNN architectures suited to CIFAR-10's 32×32 images.

The **teacher** (`create_big_model`) adds **BatchNormalization** after each convolutional layer. Batch normalization normalizes intermediate activations during training, which stabilizes learning and often allows higher learning rates. Combined with dropout, it gives the teacher strong regularization.

The **student** (`create_small_model`) is a leaner two-layer CNN with no regularization — it should learn the teacher's regularization behaviour implicitly through distillation.

Notice that the last layer of both models outputs raw **logits** (no softmax activation) — this is required for the temperature-scaled softmax used in the distillation loss.

In [ ]:
# Teacher model — larger CNN with BatchNormalization and Dropout
def create_big_model():
    tf.random.set_seed(42)
    model = keras.models.Sequential([
        # Block 1
        keras.layers.Conv2D(64, (3, 3), padding='same', input_shape=(32, 32, 3)),
        keras.layers.BatchNormalization(),
        keras.layers.Activation('relu'),
        keras.layers.MaxPooling2D((2, 2)),        # -> 16x16x64

        # Block 2
        keras.layers.Conv2D(128, (3, 3), padding='same'),
        keras.layers.BatchNormalization(),
        keras.layers.Activation('relu'),
        keras.layers.MaxPooling2D((2, 2)),        # -> 8x8x128
        keras.layers.Dropout(0.3),

        # Block 3
        keras.layers.Conv2D(256, (3, 3), padding='same'),
        keras.layers.BatchNormalization(),
        keras.layers.Activation('relu'),
        keras.layers.MaxPooling2D((2, 2)),        # -> 4x4x256
        keras.layers.Dropout(0.5),

        keras.layers.Flatten(),
        keras.layers.Dense(256, activation='relu'),
        keras.layers.Dropout(0.4),
        keras.layers.Dense(10)                   # 10 logits, no activation
    ])
    return model


# Student model — compact CNN with no regularization
def create_small_model():
    tf.random.set_seed(42)
    model = keras.models.Sequential([
        keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same',
                            input_shape=(32, 32, 3)),
        keras.layers.MaxPooling2D((2, 2)),        # -> 16x16x32

        keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        keras.layers.MaxPooling2D((2, 2)),        # -> 8x8x64

        keras.layers.Flatten(),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dense(10)                   # 10 logits, no activation
    ])
    return model

Two important things to notice:
- The last layer has **no softmax activation** because raw logits are needed for knowledge distillation (the temperature scaling is applied manually inside the `Distiller`).
- Regularization via **BatchNormalization + Dropout** is applied to the teacher but **NOT** to the student. The student should learn this regularization through the distillation process.

In [ ]:
# Create the teacher
teacher = create_big_model()

# Plot architecture
keras.utils.plot_model(teacher, rankdir="LR")

In [ ]:
# Create the student
student = create_small_model()

# Plot architecture
keras.utils.plot_model(student, rankdir="LR")

Check the actual difference in number of trainable parameters (weights and biases) between both models:

In [ ]:
# Calculates number of trainable params for a given model
def num_trainable_params(model):
    return np.sum([np.prod(v.shape) for v in model.trainable_weights])


student_params = num_trainable_params(student)
teacher_params = num_trainable_params(teacher)

print(f"Teacher model has: {teacher_params:,} trainable params.\n")
print(f"Student model has: {student_params:,} trainable params.\n")
print(f"Teacher model is roughly {teacher_params // student_params}x bigger than the student model.")


### Train the teacher

In knowledge distillation it is assumed that the teacher has already been trained, so the natural first step is to train it. You will train for **10 epochs**:

In [ ]:
# Compile the teacher model
teacher.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.Adam(),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy()]
)

# Fit the model and save the training history
teacher_history = teacher.fit(train_batches, epochs=10, validation_data=validation_batches)

In [ ]:
teacher.save('teacher_model')

## Train a student from scratch for reference

In order to assess the effectiveness of the distillation process, train a model that is identical to the student but **without** knowledge distillation. Notice that the training is done for only **7 epochs** (fewer than the teacher) to show that distillation allows quicker and better training:

In [ ]:
# Create student_scratch model with the same architecture as the distilled student
student_scratch = create_small_model()

# Compile it
student_scratch.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.Adam(),
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy()]
)

# Train and evaluate student trained from scratch
student_scratch_history = student_scratch.fit(train_batches, epochs=7, validation_data=validation_batches)

In [ ]:
student_scratch.save('student_scratch')

## Knowledge Distillation

To perform the knowledge distillation process you will use the custom `Distiller` model you previously coded.

Key parameters:
- **`alpha=0.1`**: 10% weight on the student's cross-entropy loss against ground truth, 90% on the distillation loss against the teacher's soft targets.
- **`temperature=10`**: A higher temperature produces softer probability distributions across all 10 classes, giving the student more nuanced information about class similarities (e.g. the teacher may assign small but non-zero probability to "automobile" when it sees a "truck" — the student learns from this).

The two student models are trained for only 7 epochs unlike the teacher which was trained for 10. This showcases that knowledge distillation allows for quicker training, since the student learns from an already trained model.

In [ ]:
# Create a fresh student for distillation (separate from student_scratch)
student = create_small_model()

# Create Distiller instance
distiller = Distiller(student=student, teacher=teacher)

# Compile Distiller
distiller.compile(
    student_loss_fn=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.Adam(),
    metrics=[keras.metrics.SparseCategoricalAccuracy()],
    distillation_loss_fn=keras.losses.KLDivergence(),
    alpha=0.1,
    temperature=10,
)

# Distill knowledge from teacher to student
distiller_history = distiller.fit(train_batches, epochs=7, validation_data=validation_batches)

## Comparing the models

To compare the models you can check the `sparse_categorical_accuracy` of each one on the test set:

In [ ]:
# Compute accuracies on the held-out test set
student_scratch_acc = student_scratch.evaluate(test_batches, return_dict=True).get("sparse_categorical_accuracy")
distiller_acc = distiller.evaluate(test_batches, return_dict=True).get("sparse_categorical_accuracy")
teacher_acc = teacher.evaluate(test_batches, return_dict=True).get("sparse_categorical_accuracy")

# Print results
print(f"\n\nTeacher achieved a sparse_categorical_accuracy of {teacher_acc*100:.2f}%.\n")
print(f"Student with knowledge distillation achieved a sparse_categorical_accuracy of {distiller_acc*100:.2f}%.\n")
print(f"Student without knowledge distillation achieved a sparse_categorical_accuracy of {student_scratch_acc*100:.2f}%.\n")

The teacher model yields a higher accuracy than the two student models. This is expected since it was trained for more epochs and uses a bigger architecture with BatchNormalization.

Notice that the student **without** distillation is typically outperformed by the student **with** knowledge distillation.

Since you saved the training history of each model you can create a plot for a better comparison of the two student models.

In [ ]:
# Get relevant metrics from a history object
def get_metrics(history):
    history = history.history
    acc = history['sparse_categorical_accuracy']
    val_acc = history['val_sparse_categorical_accuracy']
    return acc, val_acc


# Plot training and evaluation metrics given a dict of histories
def plot_train_eval(history_dict):
    metric_dict = {}
    for k, v in history_dict.items():
        acc, val_acc = get_metrics(v)
        metric_dict[f'{k} training acc'] = acc
        metric_dict[f'{k} eval acc'] = val_acc

    acc_plot = pd.DataFrame(metric_dict)
    acc_plot = sns.lineplot(data=acc_plot, markers=True)
    acc_plot.set_title('Training vs Evaluation Accuracy — Distilled vs Scratch Student (CIFAR-10)')
    acc_plot.set_xlabel('Epoch')
    acc_plot.set_ylabel('sparse_categorical_accuracy')
    plt.show()


# Plot comparing the two student models
plot_train_eval({
    "distilled": distiller_history,
    "student_scratch": student_scratch_history,
})

This plot is very interesting because it shows that the distilled version outperformed the unmodified one in almost all of the epochs when using the evaluation set. Alongside this, the student without distillation yields a bigger training accuracy, which is a sign that it is **overfitting** more than the distilled model. **This hints that the distilled model was able to learn from the regularization that the teacher implemented through BatchNormalization and Dropout!** Pretty cool, right?

The higher temperature (`T=10`) we used here gives the student richer information — for example, if the teacher assigns a small probability to "automobile" when looking at a "truck", the student learns that those two classes share visual features. At `T=5` (original lab), this inter-class information would be more compressed.

-----------------------------
**Congratulations on finishing this ungraded lab!** Now you should have a clearer understanding of what Knowledge Distillation is and how it can be implemented using TensorFlow and Keras.

This process is widely used for model compression and has proven to perform really well. In fact you might have heard about [`DistilBert`](https://huggingface.co/transformers/model_doc/distilbert.html), which is a smaller, faster, cheaper and lighter version of BERT.

**Keep it up!**